In [ ]:
from bokeh.io import show, output_notebook

In [ ]:
def create_data_dataframe():
    import pandas as pd
    import numpy as np
    import random
    random.seed(42)
    url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet"
    columns =  [ "tpep_pickup_datetime","tpep_dropoff_datetime","total_amount"]

    df = pd.read_parquet(url, columns = columns)

    df.rename(columns = {'tpep_pickup_datetime':'pickup_time', 'tpep_dropoff_datetime':'dropoff_time'}, inplace = True)

    return df

In [ ]:
def datasetup(df):
    import pandas as pd
    import numpy as np
    import datetime as dt

    df['pickup_time'] = pd.to_datetime(df['pickup_time'],'%Y-%m-%d %H:%M:%S')
    df['dropoff_time'] = pd.to_datetime(df['dropoff_time'],'%Y-%m-%d %H:%M:%S')

    df['pickup_hour'] = df['pickup_time'].dt.hour
    df['day_of_week'] = df['pickup_time'].dt.weekday
    df['duration'] = ((df['dropoff_time'] - df['pickup_time'])/dt.timedelta(minutes = 1))

    return df[(df['duration'] >5)]

In [ ]:
def make_bar_object(df):
    import pandas as pd
    import numpy as np
    from bokeh.models import ColumnDataSource
    from bokeh.plotting import figure

    byday = df.groupby(df['day_of_week'], as_index=False)['duration'].mean()

    weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    byday['dayname'] = byday['day_of_week'].map(lambda x: weekday_names[int(x)])

    source = ColumnDataSource(byday)

    p = figure(x_range=weekday_names, y_range=(0, 10000),width=600, height=300, tooltips=[("Duration", "@duration{0.0}")],
        title="Average Ride Duration by Day of Week")

    p.vbar(
        x='dayname',
        top='duration',
        width=0.5,
        color="green",
        legend_label="Mean Duration",
        source=source)

    return p

In [ ]:
def prepare_interactive_df(df):

    byday = df.groupby([df['day_of_week'],df['pickup_hour']])

    amount_df = byday['total_amount'].mean().unstack()

    amount_df['24'] = amount_df.iloc[:,0]

    amount_df.columns = amount_df.columns.astype(str)

    weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    amount_df['dayname'] = amount_df.index.map(lambda x: weekday_names[int(x)])

    return amount_df
prepare_interactive_df(datasetup(create_data_dataframe()))

pickup_hour,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,dayname
day_of_week,,,,,,,,,,,,,,,,,,,,,
0,30.919521,31.671244,25.753727,29.316740,29.961392,30.630228,25.623850,21.511513,20.151879,19.722028,...,21.553944,21.223335,20.674354,21.844704,23.012051,23.734493,25.205209,27.178101,30.919521,Monday
1,29.488915,27.768338,25.540982,28.956806,30.724800,27.826854,24.359065,19.937548,18.687989,18.835756,...,21.162473,20.565859,19.759852,20.704581,21.053380,21.631053,22.024742,24.031120,29.488915,Tuesday
2,24.918201,23.034259,23.230835,22.974250,28.553806,30.094547,22.155359,19.246520,18.574732,18.230331,...,21.046080,20.298274,19.632575,20.059231,20.711308,20.729526,21.191068,22.097197,24.918201,Wednesday
3,23.211866,22.797265,22.202656,22.535098,26.210990,28.556902,22.419811,18.967325,18.672870,18.603127,...,21.175902,20.665902,19.645145,20.181172,20.443797,20.279460,20.564125,21.261897,23.211866,Thursday
4,21.343214,20.306328,20.955456,21.839720,25.833766,29.508431,24.718272,20.321669,19.881742,19.919224,...,21.478775,20.634837,19.850771,19.930110,19.743530,19.706471,19.635301,19.774095,21.343214,Friday
5,20.252621,19.874471,19.891589,20.393208,22.033424,27.968026,29.240195,24.940394,22.682671,20.028899,...,20.402120,20.096195,19.371800,19.332183,20.255535,20.429136,20.646812,20.587203,20.252621,Saturday
6,20.792982,20.130326,19.376655,20.005493,22.667791,31.371735,29.902347,27.049486,26.222446,22.075972,...,21.872482,22.030727,21.339016,22.630522,24.010922,24.573795,26.772069,29.341276,20.792982,Sunday


In [ ]:
def create_layout(interactive_df):
    import numpy as np
    from bokeh.layouts import column, row
    from bokeh.models import CustomJS, Slider
    from bokeh.plotting import ColumnDataSource, figure, show, output_notebook

    source = ColumnDataSource(interactive_df)

    tooltip = ("Average Total Fare", "$@24{0,0.00}")

    weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    p = figure(x_range=weekday_names, y_range=(0, 100),width=600, height=300, tooltips=[tooltip],
        title="Dollars")

    p.vbar(
        x='dayname',
        top='24',
        width=0.5,
        color="green",
        legend_label="Mean Duration",
        source=source)

    hour = Slider(start=0, end=23, value=0, step=1, title="Hour")


    jscallback = CustomJS(args=dict(source=source, hour=hour),
                    code="""
        var data = source.data;

        data['24'] = data[this.value];

        source.change.emit();

    """)

    hour.js_on_change('value', jscallback)

    layout = row(p,hour)

    return layout

In [ ]:
output_notebook()

plot = create_layout(prepare_interactive_df(datasetup(create_data_dataframe())))
show(plot)

In [ ]:
#Specify which column you want to use for the piechart
def make_pie(data_df,column,legend_name):
    # Include the label position function
    def create_label_positions(data_df,radius):
        import numpy as np
        radius = 0.4
        label_x_pos = np.cos(data_df['angle'].cumsum()-data_df['angle'].div(2))*3*radius/4
        label_y_pos = np.sin(data_df['angle'].cumsum()-data_df['angle'].div(2))*3*radius/4
        return label_x_pos,label_y_pos

    #Imports
    import random
    import numpy as np
    import pandas as pd
    from math import pi
    import pandas as pd
    from bokeh.plotting import figure
    from bokeh.palettes import Turbo256,Category20c #The color palette
    from bokeh.models import LabelSet, ColumnDataSource
    from bokeh.transform import cumsum

    #Make a percent column and get the angle and the colors
    data_df["pct"] = data_df[column]/data_df[column].sum()
    data_df['angle'] = data_df['pct']/(data_df['pct'].sum()) * 2*pi
    colors = [Turbo256[random.randint(0,255)] for i in range(len(data_df))]
    data_df['colors'] = colors

    p = figure(height=500, title="Proportion of "+ column + " for each " + legend_name,
            tools="hover", tooltips="@"+legend_name+": @pct",
               x_range=(-0.5, 1.0))

    #The label positions and the labels and the CDS
    radius = 0.4
    data_df['label_x_pos'], data_df['label_y_pos'] = create_label_positions(data_df,radius)

    data_df["label_vals"] = np.where(data_df["pct"]>0.099,data_df[legend_name],"")
    cds = ColumnDataSource(data_df)

    labels = LabelSet(x='label_x_pos', y='label_y_pos', text="label_vals",
        angle=0, source=cds)
    p.add_layout(labels)

    #The wedges
    p.wedge(x=0, y=0, radius=radius,
        start_angle=cumsum('angle', include_zero=True), end_angle=cumsum('angle'),
        line_color="white", fill_color="colors", legend_field=legend_name, source=cds)


    p.axis.axis_label=None
    p.axis.visible=False
    p.grid.grid_line_color = None
    return p

def make_taxi_piechart(df):
    byday = df.groupby(df['day_of_week'], as_index = False)["total_amount"].sum()

    byday['pct'] = byday['total_amount']/byday['total_amount'].sum()

    weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    byday['dayname'] = byday.index.map(lambda x: weekday_names[int(x)])

    pie = make_pie(byday,'total_amount','dayname')

    return pie



In [ ]:
output_notebook()

plot = make_taxi_piechart(datasetup(create_data_dataframe()))
show(plot)